# Gaussian Mixture Models

_Classical ML_

**Soft clustering: each point partly belongs to several Gaussian blobs.**

k-means gives each point one hard label. A Gaussian Mixture Model (GMM) is gentler.

     It says the data is a blend of several bell-shaped blobs (Gaussians).

---

This notebook builds a GMM up one step at a time: first fit one on three clean blobs and read off both hard and soft cluster assignments, then turn it loose on real wine chemistry. Run each cell, read the note above it, then experiment. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

A GMM models the data as a blend of several Gaussian blobs and fits them with the EM algorithm. We build it in three steps: (1) make three clear blobs, (2) fit the GMM and read its parameters, (3) compare hard labels against soft responsibilities.

### Step 1 — Make three Gaussian blobs

We use `make_blobs` to draw 300 points from 3 well-separated clusters. Because the data is genuinely blob-shaped, a 3-component GMM should recover the clusters cleanly — a good sanity check before trying messier real data.

In [ ]:
import numpy as np
from sklearn.datasets import make_blobs
from sklearn.mixture import GaussianMixture

X, y_true = make_blobs(n_samples=300, centers=3, cluster_std=1.0, random_state=0)
print("data shape:", X.shape)

### Step 2 — Fit the GMM and read its parameters

We fit a 3-component GMM with full covariance (each blob gets its own shape). EM runs until convergence; `n_iter_` reports how many iterations it took. The fitted **mixing weights** say how much of the data each component owns, and the **means** are the blob centers it found.

In [ ]:
gmm = GaussianMixture(n_components=3, covariance_type="full", random_state=0)
gmm.fit(X)

print("converged:", gmm.converged_, "in", gmm.n_iter_, "iters")
print("mixing weights (pi):", np.round(gmm.weights_, 3))
print("component means:\n", np.round(gmm.means_, 2))

### Step 3 — Hard labels vs soft responsibilities

This is the heart of a GMM. `predict` gives each point one **hard** label (the most likely component), while `predict_proba` gives **soft** responsibilities — a probability of belonging to each component. Points near a blob's center are nearly certain; points between blobs split their membership.

In [ ]:
labels = gmm.predict(X)
print("hard label counts:", np.bincount(labels))

proba = gmm.predict_proba(X[:3])
print("soft responsibilities (first 3 points):\n", np.round(proba, 3))
print("avg log-likelihood per sample:", round(gmm.score(X), 3))

## Visualize the data & results

_Which Gaussian component generated each wine sample?_

### Step 1 — Standardize the wine data and project to 2-D

Real wine chemistry has 13 features, too many to plot. We **standardize** them so no single feature dominates, then use **PCA** to compress them into the two directions of greatest variance — giving us a flat 2-D view we can scatter-plot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture

wine = load_wine()

Xs = StandardScaler().fit_transform(wine.data)          # standardized 13 features
X2 = PCA(n_components=2, random_state=0).fit_transform(Xs)   # compressed to 2-D
print("2-D wine data shape:", X2.shape)

### Step 2 — Fit a 3-component GMM in the 2-D space

We fit the GMM directly on the 2-D PCA coordinates. `fit_predict` does both at once: it runs EM and then returns each point's hard cluster label. We also keep `gmm.means_` so we can mark where each component's center landed.

In [ ]:
gmm = GaussianMixture(n_components=3, covariance_type="full", random_state=0)
labels = gmm.fit_predict(X2)
print("cluster sizes:", np.bincount(labels))

### Step 3 — Color the points by component and mark the centers

We color each wine by its assigned component and drop a black ✕ at each Gaussian's mean. The blobs should line up with the natural groupings in the PCA scatter — a visual confirmation that the GMM found real structure in the chemistry.

In [ ]:
colors = np.array(["#4ea1ff", "#7ee787", "#c89bff"])

plt.scatter(X2[:, 0], X2[:, 1], c=colors[labels], s=20)
plt.scatter(gmm.means_[:, 0], gmm.means_[:, 1],
            c="black", marker="x", s=120)
plt.title("GMM on the Wine dataset (PCA to 2-D, 3 components)")
plt.xlabel("PCA component 1")
plt.ylabel("PCA component 2")
plt.show()